# CS781 Phase 7 — Prior Case Retrieval from the Phase 1 Legal LM, model 3 (recurrent glru weights; recurrent / attention variants; PyTorch, Kaggle GPU)
Phase 1 JAX weights → PyTorch bi-encoder (chunked mean-pooled case embeddings) → contrastive fine-tuning with in-batch and TF-IDF hard negatives → LoRMA / IA³ / partial / full fine-tuning → distillation of similarity scores into a half-depth student → optional lexical fusion → top-7 per query, one `submission` zip per experiment.

## 1. Setup and configuration

In [1]:
import os, sys, time, json, math, random, pickle, zipfile, gc, copy, glob, subprocess, importlib.util
T0 = time.time()
if importlib.util.find_spec("regex") is None: subprocess.run([sys.executable, "-m", "pip", "install", "-q", "regex"], check=False)
import numpy as np, regex as re2, torch, torch.nn as nn, torch.nn.functional as F, torch.utils.checkpoint
import multiprocessing as mp
from huggingface_hub import hf_hub_download, HfApi, login
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF"); login(token=HF_TOKEN)
except Exception as e:
    HF_TOKEN = None; print("HF login skipped:", e)

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
HF_REPO = "Exploration-Lab/CS-781-Capstone"; WORK = "/kaggle/working/cs781_phase7_m3"; os.makedirs(WORK, exist_ok=True)
PIN = None
FOUND = [(p, pickle.load(open(p, "rb"))) for p in glob.glob("/kaggle/input/**/final_params.pkl", recursive=True)]
print("Phase 1 pickles found:", [(p, x["variant"]) for p, x in FOUND])
cand = [(p, x) for p, x in FOUND if PIN in (None, p) and not any(k in x["cfg"] for k in ("encdec", "adaln"))]; del FOUND
assert cand, "no decoder-family ours_* final_params.pkl attached: add that Phase 1 run's output as a Kaggle dataset (or set PIN to its path)"
P1, d = next(((p, x) for p, x in cand if x["cfg"].get("kinds")), cand[0]); del cand
print("using", P1, "->", d["variant"])
TOKP = os.path.join(os.path.dirname(P1), "bpe_legal.json")
if not os.path.exists(TOKP): TOKP = glob.glob("/kaggle/input/**/bpe_legal.json", recursive=True)[0]
TOTAL_SEC, POST_SEC = 5.4 * 3600, 14 * 60
CT, KT, KI, BQ, HNN, TAU = 1024, 2, 8, 4, 2, 0.05
NG = torch.cuda.device_count()
EXPS = [("lorma", 1.0), ("ia3", 0.8), ("partial", 0.9), ("full", 1.1), ("kd", 1.2)]
LR = dict(lorma=3e-4, ia3=1e-3, partial=1e-4, full=3e-5, kd=5e-5)
print("gpus:", NG, "|", P1, "|", TOKP)

Phase 1 pickles found: [('/kaggle/input/datasets/nirbhaykachhatiya/phase2m3/final_params.pkl', 'ours_glru')]
using /kaggle/input/datasets/nirbhaykachhatiya/phase2m3/final_params.pkl -> ours_glru
gpus: 2 | /kaggle/input/datasets/nirbhaykachhatiya/phase2m3/final_params.pkl | /kaggle/input/datasets/nirbhaykachhatiya/phase2m3/bpe_legal.json


## 2. Data: Phase 4 CJPE files, Phase 1 byte-level BPE, tokenised once

In [2]:
PAT = re2.compile(r"'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+")
def b2u():
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256)); cs = bs[:]; n = 0
    for b in range(256):
        if b not in bs: bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))
B2U = b2u()

class BPE:
    def __init__(s, path):
        d = json.load(open(path, encoding="utf-8")); s.rank = {tuple(m): i for i, m in enumerate(d["merges"])}; s.stoi = {t: i for i, t in enumerate(d["itos"])}; s.c = {}
    def word(s, w):
        r = s.c.get(w)
        if r is None:
            x = [B2U[b] for b in w.encode("utf-8")]
            while len(x) > 1:
                best = min(zip(x, x[1:]), key=lambda q: s.rank.get(q, 1 << 60))
                if best not in s.rank: break
                a, b = best; y, i = [], 0
                while i < len(x):
                    if i < len(x) - 1 and x[i] == a and x[i + 1] == b: y.append(a + b); i += 2
                    else: y.append(x[i]); i += 1
                x = y
            r = s.c[w] = [s.stoi[t] for t in x]
        return r
    def encode(s, t): return [i for w in PAT.findall(t) for i in s.word(w)]

TK = BPE(TOKP)
from sklearn.feature_extraction.text import TfidfVectorizer
def norm(t): return " ".join(t.split())
def enc(ts): return [np.asarray(TK.encode(t), np.uint16) for t in ts]
def par(fn, xs):
    with mp.get_context("fork").Pool(os.cpu_count()) as pool: return [a for p in pool.imap(fn, [xs[i:i + 20] for i in range(0, len(xs), 20)]) for a in p]

files = HfApi().list_repo_files(HF_REPO, repo_type="dataset", token=HF_TOKEN)
def ld(name):
    fn = next(f for f in files if f.startswith("Phase_7_PCR/") and f.endswith("/" + name))
    return json.load(open(hf_hub_download(HF_REPO, fn, repo_type="dataset", token=HF_TOKEN, local_dir="/tmp/hf"), encoding="utf-8"))
def parse(obj):
    it = obj.items() if isinstance(obj, dict) else [(o.get("id", o.get("case_id", o.get("doc_id"))), o) for o in obj]; out = {}
    for k, v in it:
        t, rel = (v.get("text", v.get("document", "")), v.get("relevant_candidates")) if isinstance(v, dict) else (v, None)
        out[str(k)] = (norm(" ".join(map(str, t)) if isinstance(t, list) else str(t)), None if rel is None else sorted({str(r) for r in rel}))
    return out

TOKC = f"{WORK}/data.pkl"
if os.path.exists(TOKC): DS = pickle.load(open(TOKC, "rb"))
else:
    DS = {}; t0 = time.time()
    for sp in ("train", "dev", "test"):
        raw = ld(f"{sp}.json"); print(sp, "query file:", type(raw).__name__, "| first keys:", list(next(iter(raw.values())).keys()) if isinstance(raw, dict) and isinstance(next(iter(raw.values())), dict) else (list(raw[0].keys()) if isinstance(raw, list) else "n/a"))
        Q, Pl = parse(raw), parse(ld(f"{sp}_candidates.json")); qid, pid = list(Q), list(Pl); pidx = {p: i for i, p in enumerate(pid)}
        vec = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.5, max_features=300000, dtype=np.float32).fit([Pl[p][0] for p in pid])
        lex = (vec.transform([Q[q][0] for q in qid]) @ vec.transform([Pl[p][0] for p in pid]).T).toarray().astype(np.float32)
        toks = par(enc, [Q[q][0] for q in qid] + [Pl[p][0] for p in pid])
        gold = [[pidx[r] for r in Q[q][1] if r in pidx] if Q[q][1] is not None else None for q in qid]; gsz = [len(Q[q][1]) if Q[q][1] is not None else 0 for q in qid]
        DS[sp] = dict(qid=qid, pid=pid, pidx=pidx, qtok=toks[:len(qid)], ptok=toks[len(qid):], gold=gold, gsz=gsz, lex=lex)
        if sp == "train":
            DS[sp]["hn"] = [[int(j) for j in np.argsort(-lex[i])[:60] if j not in set(g)][:30] for i, g in enumerate(gold)]
        del Q, Pl, vec
    pickle.dump(DS, open(TOKC, "wb")); gc.collect(); print(f"prepared in {time.time()-t0:.0f}s")
for sp, x in DS.items(): print(f"{sp}: {len(x['qid'])} queries, pool {len(x['pid'])}, gold/query {np.mean(x['gsz']):.2f}, gold inside pool {np.mean([len(g) / max(s, 1) for g, s in zip(x['gold'], x['gsz'])]) if sp != 'test' else float('nan'):.2f}, median tokens q/pool {np.median([len(t) for t in x['qtok']]):.0f}/{np.median([len(t) for t in x['ptok']]):.0f}")

def selk(nc, rnd, K):
    if nc <= K: return list(range(nc))
    return sorted(random.sample(range(nc), K)) if rnd else np.unique(np.linspace(0, nc - 1, K).round().astype(int)).tolist()

def mkc(toks, K, rnd):
    ids = np.zeros((len(toks), K, CT), np.int64); ln = np.zeros((len(toks), K), np.int64)
    for b, t in enumerate(toks):
        n = len(t)
        for k, c in enumerate(selk(max(1, -(-n // CT)), rnd, K)):
            x = t[min(c * CT, max(n - CT, 0)):][:CT]; ids[b, k, :len(x)] = x; ln[b, k] = len(x)
    return torch.from_numpy(ids).cuda(), torch.from_numpy(ln).cuda()

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

train query file: dict | first keys: ['text', 'relevant_candidates']


Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

... (4 outputs left out)

test query file: dict | first keys: ['text']


Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

prepared in 104s
train: 827 queries, pool 4320, gold/query 6.42, gold inside pool 0.99, median tokens q/pool 5805/4690
dev: 118 queries, pool 1023, gold/query 8.25, gold inside pool 1.00, median tokens q/pool 5830/5220
test: 237 queries, pool 1727, gold/query 0.00, gold inside pool nan, median tokens q/pool 6068/5043


## 3. Model: Phase 1 decoder ported to PyTorch, chunk encoder + attention pooling head, adaptation modules

In [3]:
P, C = d["params"], d["cfg"]; assert C["norm"] == "rms" and C["mlp"] == "swiglu", d["variant"]
if C.get("kinds"): KINDS, BL = C["kinds"], P["blocks"]
else: KINDS, BL = "A" * P["layers"]["wq"].shape[0], [{k: v[i] for k, v in P["layers"].items()} for i in range(P["layers"]["wq"].shape[0])]
NB = len(KINDS); D, V = P["emb"].shape[1], P["emb"].shape[0]; ia, ir = KINDS.find("A"), KINDS.find("R"); SSM_K, DR = BL[ir]["conv"].shape if ir >= 0 else (4, D)
HD = BL[ia]["qn"].shape[0] if ia >= 0 else 64; H = BL[ia]["wq"].shape[1] // HD if ia >= 0 else 1; KV = BL[ia]["wk"].shape[1] // HD if ia >= 0 else 1; R = max(H // KV, 1)
WIN = 256; LOC = [bool(C.get("hybrid")) and i % 3 != 2 for i in range(NB)]; ALL = ("wq", "wk", "wv", "wx", "wg", "wa", "wo", "w1", "w3", "w2")
INV = torch.tensor(1.0 / (10000.0 ** (np.arange(0, HD, 2) / HD)), dtype=torch.float32)
print(d["variant"], f"kinds={KINDS} D={D} H={H} KV={KV} HD={HD} conv={SSM_K} V={V}", "|", torch.cuda.get_device_name(0))

def f32(w): return torch.from_numpy(np.asarray(w, np.float32).copy())
def lin(w):
    m = nn.Linear(w.shape[0], w.shape[1], bias=False); m.weight.data = torch.from_numpy(np.ascontiguousarray(w.T, dtype=np.float32)); return m

class RMS(nn.Module):
    def __init__(s, w): super().__init__(); s.w = nn.Parameter(f32(w))
    def forward(s, x): x = x.float(); return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * s.w

def rope(x, cos, sin):
    a, b = x[..., :HD // 2].float(), x[..., HD // 2:].float()
    return torch.cat([a * cos - b * sin, b * cos + a * sin], -1)

def scan(a, b, c=16):
    with torch.autocast("cuda", enabled=False):
        N, T, Dr = a.shape; nc = T // c; assert T % c == 0
        A = a.float().clamp_min(math.exp(-5)).log().view(N, nc, c, Dr).cumsum(2)
        S = A.exp() * (b.float().view(N, nc, c, Dr) * (-A).exp()).cumsum(2)
        ca = A[:, :, -1].cumsum(1); tri = torch.ones(nc, nc, dtype=torch.bool, device=a.device).tril()
        W = (ca[:, :, None] - ca[:, None]).masked_fill(~tri[None, :, :, None], -1e4).exp()
        ce = torch.einsum("nkjd,njd->nkd", W, S[:, :, -1]); cin = F.pad(ce, (0, 0, 1, 0))[:, :-1]
        return (S + A.exp() * cin[:, :, None]).view(N, T, Dr)

class ABlk(nn.Module):
    def __init__(s, lp, loc):
        super().__init__(); s.loc = loc
        for n in ("wq", "wk", "wv", "wo", "w1", "w3", "w2"): setattr(s, n, lin(lp[n]))
        s.n1, s.n2, s.qn, s.kn = RMS(lp["n1"]), RMS(lp["n2"]), RMS(lp["qn"]), RMS(lp["kn"]); s.g = nn.Parameter(f32(lp["g"])) if "g" in lp else None
    def gate(s, j): return 1.0 if s.g is None else s.g[j]
    def qkv(s, h, cos, sin):
        B, T, _ = h.shape; q, k, v = s.wq(h).view(B, T, H, HD), s.wk(h).view(B, T, KV, HD), s.wv(h).view(B, T, KV, HD)
        return rope(s.qn(q), cos, sin).to(v.dtype), rope(s.kn(k), cos, sin).to(v.dtype), v
    def out(s, x, o):
        x = x + s.gate(0) * s.wo(o); h = s.n2(x); return x + s.gate(1) * s.w2(F.silu(s.w1(h)) * s.w3(h))
    def attn(s, q, k, v, mask, causal): return F.scaled_dot_product_attention(q.transpose(1, 2), k.repeat_interleave(R, 2).transpose(1, 2), v.repeat_interleave(R, 2).transpose(1, 2), attn_mask=mask, is_causal=causal)
    def forward(s, x, cos, sin, lm):
        q, k, v = s.qkv(s.n1(x), cos, sin); B, T = x.shape[:2]
        return s.out(x, s.attn(q, k, v, lm if s.loc else None, not s.loc).transpose(1, 2).reshape(B, T, H * HD))
    def prefill(s, x, cos, sin, am, ok):
        q, k, v = s.qkv(s.n1(x), cos, sin); B, T = x.shape[:2]
        return s.out(x, s.attn(q, k, v, am, False).transpose(1, 2).reshape(B, T, H * HD)), (k.transpose(1, 2), v.transpose(1, 2))
    def expand(s, c, nb, Tm):
        k, v = c; kc, vc = k.new_zeros(k.shape[0] * nb, KV, Tm, HD), v.new_zeros(v.shape[0] * nb, KV, Tm, HD)
        kc[:, :, :k.shape[2]] = k.repeat_interleave(nb, 0); vc[:, :, :k.shape[2]] = v.repeat_interleave(nb, 0); return kc, vc
    def step(s, x, cos, sin, c, t, valid):
        kc, vc = c; n = x.shape[0]; q, k, v = s.qkv(s.n1(x), cos, sin); kc[:, :, t] = k[:, 0]; vc[:, :, t] = v[:, 0]; lo = max(0, t + 1 - WIN) if s.loc else 0
        a = (q[:, 0].view(n, KV, R, HD) @ kc[:, :, lo:t + 1].transpose(-1, -2)) * HD ** -0.5
        a = a.masked_fill(~valid[:, None, None, lo:t + 1], float("-inf")).float().softmax(-1).to(v.dtype)
        return s.out(x, (a @ vc[:, :, lo:t + 1]).reshape(n, 1, H * HD)), c

class RBlk(nn.Module):
    loc = False
    def __init__(s, lp):
        super().__init__()
        for n in ("wx", "wg", "wa", "wo", "w1", "w3", "w2"): setattr(s, n, lin(lp[n]))
        s.n1, s.n2, s.ab, s.conv = RMS(lp["n1"]), RMS(lp["n2"]), nn.Parameter(f32(lp["ab"])), nn.Parameter(f32(lp["conv"]))
    def mlp(s, x): h = s.n2(x); return x + s.w2(F.silu(s.w1(h)) * s.w3(h))
    def run(s, x, ok=None):
        T = x.shape[1]; h = s.n1(x); xv = s.wx(h).float()
        if ok is not None: xv = xv * ok[..., None]
        xp = F.pad(xv, (0, 0, SSM_K - 1, 0)); u = sum(xp[:, j:j + T] * s.conv[j] for j in range(SSM_K)); a = torch.sigmoid(s.wa(h).float() + s.ab); b = (1 - a) * u
        if ok is not None: a = torch.where(ok[..., None], a, torch.ones_like(a)); b = b * ok[..., None]
        sv = scan(a, b); return s.mlp(x + s.wo(sv * F.silu(s.wg(h).float()))), sv[:, -1], xv[:, -(SSM_K - 1):]
    def core(s, x): return s.run(x)[0]
    def forward(s, x, cos, sin, lm): return torch.utils.checkpoint.checkpoint(s.core, x, use_reentrant=False) if s.training and torch.is_grad_enabled() else s.core(x)
    def prefill(s, x, cos, sin, am, ok): x, st, buf = s.run(x, ok); return x, (buf, st)
    def expand(s, c, nb, Tm): return tuple(x.repeat_interleave(nb, 0) for x in c)
    def step(s, x, cos, sin, c, t, valid):
        buf, st = c; h = s.n1(x); win = torch.cat([buf, s.wx(h).float()], 1); u = (win * s.conv[None]).sum(1)
        a = torch.sigmoid(s.wa(h).float()[:, 0] + s.ab); st = a * st + (1 - a) * u
        return s.mlp(x + s.wo((st * F.silu(s.wg(h).float()[:, 0]))[:, None])), (win[:, 1:], st)

class Enc(nn.Module):
    def __init__(s, keep):
        super().__init__(); s.emb = nn.Embedding(V, D); s.emb.weight.data = f32(P["emb"]); s.nf = RMS(P["nf"])
        s.blocks = nn.ModuleList([ABlk(BL[i], LOC[i]) if KINDS[i] == "A" else RBlk(BL[i]) for i in keep])
        t = torch.arange(CT)[:, None] * INV[None]; s.register_buffer("cos", t.cos(), persistent=False); s.register_buffer("sin", t.sin(), persistent=False)
    def forward(s, ids):
        T = ids.shape[1]; x = s.emb(ids); cos, sin = s.cos[:T][None, :, None], s.sin[:T][None, :, None]
        lm = torch.ones(T, T, dtype=torch.bool, device=ids.device).tril().triu(1 - WIN)
        for b in s.blocks: x = b(x, cos, sin, lm)
        return s.nf(x)

class DocEnc(nn.Module):
    def __init__(s, enc): super().__init__(); s.enc = enc
    def forward(s, ids, ln):
        B, Kc, T = ids.shape; h = s.enc(ids.view(B * Kc, T)).float().view(B, Kc, T, -1)
        tm = (torch.arange(T, device=ids.device)[None, None] < ln[..., None]).float(); hc = (h * tm[..., None]).sum(2) / ln.clamp(min=1)[..., None]; cm = (ln > 0).float()[..., None]
        return F.normalize((hc * cm).sum(1) / cm.sum(1).clamp(min=1), dim=-1)

class LoRMA(nn.Module):
    def __init__(s, b, r=16): super().__init__(); s.b = b; o = b.out_features; s.A = nn.Parameter(torch.randn(o, r) * o ** -0.5); s.B = nn.Parameter(torch.zeros(r, o))
    def forward(s, x): y = s.b(x); return y + (y @ s.A) @ s.B

class Scale(nn.Module):
    def __init__(s, b): super().__init__(); s.b = b; s.l = nn.Parameter(torch.ones(b.out_features))
    def forward(s, x): y = s.b(x); return y * s.l.to(y.dtype)

class ScaleIn(nn.Module):
    def __init__(s, b): super().__init__(); s.b = b; s.l = nn.Parameter(torch.ones(b.in_features))
    def forward(s, x): return s.b(x * s.l)

def adapt(m, kind):
    if kind in ("full", "kd"): return m
    for p in m.parameters(): p.requires_grad = False
    if kind == "partial":
        for p in [*m.enc.nf.parameters(), *[q for b in m.enc.blocks[-4:] for q in b.parameters()]]: p.requires_grad = True
        return m
    for b in m.enc.blocks:
        if kind == "lorma":
            for n in ALL:
                l = getattr(b, n, None)
                if isinstance(l, nn.Linear): setattr(b, n, LoRMA(l))
        elif kind == "ia3":
            for n in (("wk", "wv") if isinstance(b, ABlk) else ("wx", "wg")): setattr(b, n, Scale(getattr(b, n)))
            b.w2 = ScaleIn(b.w2)
    return m

def build(kind): return adapt(DocEnc(Enc(range(0, NB, 2) if kind == "kd" else range(NB))), kind)

ours_glru kinds=RRRRRRRRRRRR D=768 H=1 KV=1 HD=64 conv=4 V=16384 | Tesla T4


## 4. Training utilities (DataParallel, fp16 autocast, class-weighted loss, time-budgeted cosine schedule)

In [4]:
LAMS = (0.0, 0.25, 0.5, 1.0, 2.0)
def zs(S): return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)
def retrieve(sp, Eq, Ep, lam):
    x = DS[sp]; S = zs(x["lex"]) * lam
    if Eq is not None: S = S + zs((Eq @ Ep.T).numpy())
    for i, q in enumerate(x["qid"]):
        j = x["pidx"].get(q)
        if j is not None: S[i, j] = -1e9
    return np.argsort(-S, 1)[:, :7]
def f7(sp, top):
    x = DS[sp]; tp = sum(len(set(t.tolist()) & set(g)) for t, g in zip(top, x["gold"])); P_, R_ = tp / (7 * len(top)), tp / max(sum(x["gsz"]), 1)
    return dict(f1=2 * P_ * R_ / (P_ + R_) if tp else 0.0, p=P_, r=R_)
def best_lam(sp, Eq, Ep):
    tab = {l: f7(sp, retrieve(sp, Eq, Ep, l)) for l in LAMS}; l = max(tab, key=lambda k: tab[k]["f1"]); return l, tab

@torch.no_grad()
def embed(dm, toks, K=None, bs=16):
    K = K or KI; tr = dm.training; dm.eval(); out = []
    for i in range(0, len(toks), bs):
        ids, ln = mkc(toks[i:i + bs], K, False)
        with torch.autocast("cuda", torch.float16): out.append(dm(ids, ln).float().cpu())
    dm.train(tr); return torch.cat(out)

def embed_split(dm, sp, K=None): return embed(dm, DS[sp]["qtok"], K), embed(dm, DS[sp]["ptok"], K)

def fit(name, m, kind, budget, teacher=None):
    m.cuda().train(); dm = nn.DataParallel(m) if NG > 1 else m; tdm = (nn.DataParallel(teacher) if NG > 1 else teacher) if teacher is not None else None
    tp = [p for p in m.parameters() if p.requires_grad]; opt = torch.optim.AdamW(tp, lr=LR[kind], weight_decay=0.01, betas=(0.9, 0.98)); sc = torch.amp.GradScaler("cuda")
    tr = DS["train"]; ok = [i for i, g in enumerate(tr["gold"]) if g and len(tr["hn"][i]) >= HNN]; labels = torch.arange(BQ).cuda()
    step, total, tb, ema, hist, t0 = 0, 10 ** 9, None, None, [], time.time(); torch.cuda.reset_peak_memory_stats()
    while step < total and time.time() - T0 < TOTAL_SEC - POST_SEC:
        qs = random.sample(ok, BQ); qt, pt, nt, did = [], [], [], []
        for qi in qs: p = random.choice(tr["gold"][qi]); did.append(p); qt.append(tr["qtok"][qi]); pt.append(tr["ptok"][p])
        for qi in qs:
            for j in random.sample(tr["hn"][qi], HNN): did.append(j); nt.append(tr["ptok"][j])
        gs = [set(tr["gold"][qi]) for qi in qs]; mask = torch.tensor([[(did[j] in gs[i]) and j != i for j in range(len(did))] for i in range(BQ)]).cuda()
        ids, ln = mkc(qt + pt + nt, KT, True)
        f = min(1.0, (step + 1) / 50) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(step / total, 1.0))))
        for g in opt.param_groups: g["lr"] = LR[kind] * f
        with torch.autocast("cuda", torch.float16):
            E = dm(ids, ln).float()
            if tdm is not None:
                with torch.no_grad(): Et = tdm(ids, ln).float()
        S = (E[:BQ] @ E[BQ:].T / TAU).masked_fill(mask, -1e4); loss = F.cross_entropy(S, labels)
        if tdm is not None: St = (Et[:BQ] @ Et[BQ:].T / TAU).masked_fill(mask, -1e4); loss = 0.5 * loss + 0.5 * F.kl_div(F.log_softmax(S, -1), F.softmax(St, -1), reduction="batchmean")
        opt.zero_grad(set_to_none=True); sc.scale(loss).backward(); sc.unscale_(opt); nn.utils.clip_grad_norm_(tp, 1.0); sc.step(opt); sc.update(); step += 1
        if total == 10 ** 9:
            if step == 5: torch.cuda.synchronize(); tb = time.time()
            if step == 25:
                torch.cuda.synchronize(); rate = (time.time() - tb) / 20; total = step + int(0.97 * (budget - (time.time() - t0)) / rate)
                print(f"[{name}] {rate:.3f}s/step -> total_steps={total} (~{total * BQ / len(ok):.1f} epochs over {len(ok)} queries)", flush=True)
        if step % 10 == 0: ema = loss.item() if ema is None else 0.9 * ema + 0.1 * loss.item()
        if step % 50 == 0 and ema is not None: hist.append((step, ema))
        if step % 200 == 0: print(f"[{name}] step {step}/{total} loss {ema:.4f} | {(time.time() - t0) / 60:.1f} min", flush=True)
    tt = time.time() - t0; mem = torch.cuda.max_memory_allocated() / 2 ** 30; del opt, sc; gc.collect(); torch.cuda.empty_cache(); m.eval()
    t1 = time.time(); Eq, Ep = embed_split(dm, "dev"); lam, tab = best_lam("dev", Eq, Ep); es = time.time() - t1; Eqt, Ept = embed_split(dm, "test")
    top = retrieve("test", Eqt, Ept, lam); out = {DS["test"]["qid"][i]: [DS["test"]["pid"][j] for j in t] for i, t in enumerate(top)}
    assert all(len(set(v)) == 7 for v in out.values()) and len(out) == len(DS["test"]["qid"])
    os.makedirs(f"{WORK}/{name}", exist_ok=True); pj = f"{WORK}/{name}/prediction.json"; json.dump(out, open(pj, "w"))
    with zipfile.ZipFile(f"{WORK}/sub_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, "prediction.json")
    np.save(f"{WORK}/{name}/dev_q.npy", Eq.half().numpy()); np.save(f"{WORK}/{name}/dev_p.npy", Ep.half().numpy()); torch.save({n: p.detach().cpu() for n, p in m.named_parameters() if p.requires_grad}, f"{WORK}/{name}/weights.pt")
    res = dict(name=name, f1=tab[lam]["f1"], p=tab[lam]["p"], r=tab[lam]["r"], lam=lam, f1_dense=tab[0.0]["f1"], lam_table={str(k): v["f1"] for k, v in tab.items()}, params_M=sum(p.numel() for p in m.parameters()) / 1e6,
               train_M=sum(p.numel() for p in tp) / 1e6, steps=step, train_min=tt / 60, mem_GB=mem, embed_s=es, hist=hist)
    print(f"[{name}] dev F1@7 {res['f1']:.4f} (P {res['p']:.4f} R {res['r']:.4f}) at lambda={lam} | dense-only {res['f1_dense']:.4f} | embed dev {es:.0f}s", flush=True); return res, m

## 5. Experiments (time-shared budget; KD teacher = best dev macro-F1 model so far, student = every-other-layer Phase 1 init)

In [5]:
z = DocEnc(Enc(range(NB))).cuda().eval(); zdm = nn.DataParallel(z) if NG > 1 else z; Eq, Ep = embed_split(zdm, "dev"); zl, ztab = best_lam("dev", Eq, Ep); lex = f7("dev", retrieve("dev", None, None, 1.0))
print(f"BASELINES on dev (F1@7): TF-IDF only {lex['f1']:.4f} | Phase 1 zero-shot dense {ztab[0.0]['f1']:.4f} | zero-shot + TF-IDF fusion {ztab[zl]['f1']:.4f} (lambda {zl})", flush=True)
ZS = dict(tfidf=lex["f1"], dense=ztab[0.0]["f1"], fused=ztab[zl]["f1"]); json.dump(ZS, open(f"{WORK}/baselines.json", "w")); del z, zdm, Eq, Ep; gc.collect(); torch.cuda.empty_cache()
RES, BEST = {}, (-1.0, None)
for i, (name, w) in enumerate(EXPS):
    alloc = (TOTAL_SEC - (time.time() - T0)) * w / sum(x[1] for x in EXPS[i:])
    if alloc < 25 * 60: print("skip", name, "(not enough time)"); continue
    print(f"=== {name}: {alloc / 60:.0f} min allocated, {(time.time() - T0) / 60:.0f} min elapsed", flush=True)
    assert name != "kd" or BEST[1] is not None
    tc = BEST[1].cuda().eval() if name == "kd" else None
    if tc is not None:
        for p in tc.parameters(): p.requires_grad = False
    RES[name], m = fit(name, build(name), name, alloc - POST_SEC, tc)
    if name != "kd" and RES[name]["f1"] > BEST[0]: BEST = (RES[name]["f1"], copy.deepcopy(m).cpu())
    if tc is not None: tc.cpu()
    m.cpu(); del m, tc; gc.collect(); torch.cuda.empty_cache(); json.dump(RES, open(f"{WORK}/results.json", "w"), indent=1)

print(f"{'exp':9s}{'total M':>9s}{'train M':>9s}{'F1@7':>8s}{'P@7':>8s}{'R@7':>8s}{'lam':>6s}{'dense F1':>10s}{'min':>7s}{'GB':>6s}{'embed s':>9s}")
for k, r in RES.items(): print(f"{k:9s}{r['params_M']:9.1f}{r['train_M']:9.2f}{r['f1']:8.4f}{r['p']:8.4f}{r['r']:8.4f}{r['lam']:6.2f}{r['f1_dense']:10.4f}{r['train_min']:7.1f}{r['mem_GB']:6.1f}{r['embed_s']:9.0f}")
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
plt.figure(figsize=(9, 5))
for k, r in RES.items():
    if r["hist"]: plt.plot(*zip(*r["hist"]), label=k)
plt.xlabel("step"); plt.ylabel("train loss (EMA)"); plt.grid(True); plt.legend(); plt.savefig(f"{WORK}/loss_curves.png", dpi=200, bbox_inches="tight"); plt.show()
print("submissions:", sorted(glob.glob(f"{WORK}/sub_*.zip")))

BASELINES on dev (F1@7): TF-IDF only 0.3558 | Phase 1 zero-shot dense 0.2401 | zero-shot + TF-IDF fusion 0.3558 (lambda 0.5)
=== lorma: 64 min allocated, 4 min elapsed
[lorma] 2.251s/step -> total_steps=1291 (~6.3 epochs over 817 queries)
[lorma] step 200/1291 loss 1.9521 | 7.4 min
[lorma] step 400/1291 loss 1.7990 | 14.9 min
[lorma] step 600/1291 loss 1.4494 | 22.3 min
[lorma] step 800/1291 loss 1.6951 | 29.7 min
[lorma] step 1000/1291 loss 1.4339 | 37.2 min
[lorma] step 1200/1291 loss 1.6305 | 44.6 min
[lorma] dev F1@7 0.3936 (P 0.4286 R 0.3638) at lambda=0.5 | dense-only 0.3646 | embed dev 178s
=== ia3: 53 min allocated, 61 min elapsed
[ia3] 2.056s/step -> total_steps=1095 (~5.4 epochs over 817 queries)
[ia3] step 200/1095 loss 1.6864 | 6.8 min
[ia3] step 400/1095 loss 2.1241 | 13.6 min
[ia3] step 600/1095 loss 1.8290 | 20.5 min
[ia3] step 800/1095 loss 2.0596 | 27.3 min
[ia3] step 1000/1095 loss 2.0687 | 34.1 min
[ia3] dev F1@7 0.3613 (P 0.3935 R 0.3340) at lambda=2.0 | dense-only 

## 6. Long-document analysis: which part of the judgment carries the signal (best non-KD model, dev set)

In [6]:
bm = BEST[1].cuda().eval(); dm = nn.DataParallel(bm) if NG > 1 else bm
print("pooling over chunks: dev F1@7 (dense only / best fusion)")
for K_ in (1, 4, KI):
    Eq, Ep = embed_split(dm, "dev", K_); l, tab = best_lam("dev", Eq, Ep); print(f"  {K_} chunks/doc: {tab[0.0]['f1']:.4f} / {tab[l]['f1']:.4f} (lambda {l})")
Eq, Ep = embed_split(dm, "dev"); l, tab = best_lam("dev", Eq, Ep); print("lambda sweep:", {k: round(v["f1"], 4) for k, v in tab.items()})
top = retrieve("dev", Eq, Ep, l); x = DS["dev"]; hits = np.array([len(set(t.tolist()) & set(g)) for t, g in zip(top, x["gold"])]); gs = np.array(x["gsz"]); ql = np.array([len(t) for t in x["qtok"]])
print("queries by hits@7:", np.bincount(hits, minlength=8).tolist(), "| zero-hit share", round(float((hits == 0).mean()), 3))
for lo, hi in ((0, 3), (3, 6), (6, 10), (10, 10 ** 6)):
    s_ = (gs >= lo) & (gs < hi)
    if s_.any(): print(f"  gold size [{lo},{hi}): {int(s_.sum())} queries, recall@7 {hits[s_].sum() / max(gs[s_].sum(), 1):.3f}")
for nm, s_ in (("short queries (<= median length)", ql <= np.median(ql)), ("long queries (> median length)", ql > np.median(ql))): print(f"  {nm}: precision@7 {hits[s_].sum() / (7 * max(int(s_.sum()), 1)):.3f}")
bm.cpu()

pooling over chunks: dev F1@7 (dense only / best fusion)
  1 chunks/doc: 0.2968 / 0.3780 (lambda 1.0)
  4 chunks/doc: 0.3580 / 0.3947 (lambda 0.5)
  8 chunks/doc: 0.3646 / 0.3936 (lambda 0.5)
lambda sweep: {0.0: 0.3646, 0.25: 0.3902, 0.5: 0.3936, 1.0: 0.3902, 2.0: 0.3791}
queries by hits@7: [7, 24, 27, 18, 9, 18, 9, 6] | zero-hit share 0.059
  gold size [0,3): 33 queries, recall@7 0.765
  gold size [3,6): 30 queries, recall@7 0.583
  gold size [6,10): 25 queries, recall@7 0.544
  gold size [10,1000000): 30 queries, recall@7 0.232
  short queries (<= median length): precision@7 0.332
  long queries (> median length): precision@7 0.525


DocEnc(
  (enc): Enc(
    (emb): Embedding(16384, 768)
    (nf): RMS()
    (blocks): ModuleList(
      (0-11): 12 x RBlk(
        (wx): LoRMA(
          (b): Linear(in_features=768, out_features=768, bias=False)
        )
        (wg): LoRMA(
          (b): Linear(in_features=768, out_features=768, bias=False)
        )
        (wa): LoRMA(
          (b): Linear(in_features=768, out_features=768, bias=False)
        )
        (wo): LoRMA(
          (b): Linear(in_features=768, out_features=768, bias=False)
        )
        (w1): LoRMA(
          (b): Linear(in_features=768, out_features=2048, bias=False)
        )
        (w3): LoRMA(
          (b): Linear(in_features=768, out_features=2048, bias=False)
        )
        (w2): LoRMA(
          (b): Linear(in_features=2048, out_features=768, bias=False)
        )
        (n1): RMS()
        (n2): RMS()
      )
    )
  )
)